# End-to-End Workflow Create Model Group Deployment

### Load Dependencies

In [ ]:
from __future__ import annotations

import importlib
import json
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict

from ads.model import GenericModel

try:
    import cloudpickle as serializer
except ImportError:  # pragma: no cover
    import pickle as serializer

import argparse
import json
import logging
import os
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, Optional

import ads
import oci
import requests
from ads.model import ModelVersionSet
from oci.signer import Signer


### Define Constants

In [ ]:
PROJECT_OCID = 'ocid1.datascienceproject...'
COMPARTMENT_OCID = 'ocid1.compartment...'
LOG_GROUP_OCID = 'ocid1.loggroup...'
LOG_OCID = 'ocid1.log...'
INFERENCE_CONDA_ENV = "generalml_p311_cpu_x86_64_v1"
INFERENCE_PYTHON_VERSION = "3.11"
TEMPLATES_DIR = '<full-working-directory-path>/complete-workflow-example/templates'
DEFAULT_PROFILE = "DEFAULT"
PROJECT_ROOT_PATH = '<full-working-directory-path>/'

### Utility Functions

In [3]:

logger = logging.getLogger("deploy_to_oci")

In [4]:
def configure_logging(verbose: bool = False) -> None:
    logging.basicConfig(
        level=logging.DEBUG if verbose else logging.INFO,
        format="%(asctime)s %(levelname)s %(name)s - %(message)s",
    )

In [5]:
def serialize_model_instance(artifact_dir: Path, package_name: str, class_name: str):
    importlib.invalidate_caches()
    sys.path.insert(0, str(artifact_dir))
    try:
        module = importlib.import_module(f"{package_name}.model")
        model_class = getattr(module, class_name)
        model_instance = model_class()
        target_path = artifact_dir / "model.pickle"
        with target_path.open("wb") as file_handle:
            serializer.dump(model_instance, file_handle)
        print('model serialised successfully')
    except Exception as e:
        print(e)

In [6]:
def wait_for_state(
    label: str,
    getter: Callable[[], Any],
    success_states: Iterable[str],
    failure_states: Optional[Iterable[str]] = None,
    timeout_seconds: int = 20000,
    interval_seconds: int = 20,
):
    success = {state.upper() for state in success_states}
    failure = {state.upper() for state in (failure_states or [])}
    deadline = time.time() + timeout_seconds

    while True:
        resource = getter()
        state = (getattr(resource, "lifecycle_state", None) or "UNKNOWN").upper()
        logger.info("%s lifecycle_state=%s", label, state)
        if state in success:
            return resource
        if state in failure:
            raise RuntimeError(f"{label} entered failure state {state}")
        if time.time() >= deadline:
            raise TimeoutError(f"Timed out waiting for {label} to reach one of {sorted(success)}")
        time.sleep(interval_seconds)

In [7]:
def create_model_group(
    client: oci.data_science.DataScienceClient,
    display_name: str,
    description: str,
    members: list[dict],
    COMPARTMENT_OCID,
    PROJECT_OCID,
    
) -> str:
    response = client.create_model_group(
        create_base_model_group_details=oci.data_science.models.CreateModelGroupDetails(
            create_type="CREATE",
            compartment_id=COMPARTMENT_OCID,
            project_id=PROJECT_OCID,
            model_group_details=oci.data_science.models.HomogeneousModelGroupDetails(type="HOMOGENEOUS"),
            member_model_entries=oci.data_science.models.MemberModelEntries(
                member_model_details=[
                    oci.data_science.models.MemberModelDetails(
                        model_id=member["model_id"],
                        inference_key=member["inference_key"],
                    )
                    for member in members
                ]
            ),
            display_name=display_name,
            description=description,
        )
    )
    model_group_id = response.data.id
    wait_for_state(
        label=f"model group {display_name}",
        getter=lambda: client.get_model_group(model_group_id).data,
        success_states={"ACTIVE"},
        failure_states={"FAILED", "DELETED"},
    )
    return model_group_id

In [8]:
def upload_model_group_artifact(
    client: oci.data_science.DataScienceClient,
    model_group_id: str,
    zip_path: Path,
) -> None:
    content_disposition = f"attachment;filename={zip_path.name}"
    with zip_path.open("rb") as artifact_file:
        client.create_model_group_artifact(
            model_group_id=model_group_id,
            model_group_artifact=artifact_file,
            content_disposition=content_disposition,
        )
    wait_for_state(
        label=f"model group artifact {model_group_id}",
        getter=lambda: client.get_model_group(model_group_id).data,
        success_states={"ACTIVE"},
        failure_states={"FAILED", "DELETED"},
    )

In [9]:
def create_model_deployment(
    client: oci.data_science.DataScienceClient,
    display_name: str,
    description: str,
    model_group_id: str, LOG_GROUP_OCID : str, LOG_OCID : str, COMPARTMENT_OCID : str, PROJECT_OCID : str,
) -> oci.data_science.models.ModelDeployment:
    try:
        print('setting config')
        instance_shape_config_details = oci.data_science.models.ModelDeploymentInstanceShapeConfigDetails(
            memory_in_gbs=16,
            ocpus=1,
        )

        print('setting compute')
        instance_configuration = oci.data_science.models.InstanceConfiguration(
            instance_shape_name="VM.Standard.E4.Flex",
            model_deployment_instance_shape_config_details=instance_shape_config_details,
        )

        print('setting scaling policy')
        scaling_policy = oci.data_science.models.FixedSizeScalingPolicy(
            policy_type="FIXED_SIZE",
            instance_count=1,
        )
        infrastructure_config_details = oci.data_science.models.InstancePoolInfrastructureConfigurationDetails(
            infrastructure_type="INSTANCE_POOL",
            instance_configuration=instance_configuration,
            scaling_policy=scaling_policy,
        )

        print('setting concurrency')
        environment_config_details = oci.data_science.models.DefaultModelDeploymentEnvironmentConfigurationDetails(
            environment_configuration_type="DEFAULT",
            environment_variables={"WEB_CONCURRENCY": "1"},
        )
        model_group_config_details = oci.data_science.models.ModelGroupConfigurationDetails(
            model_group_id=model_group_id,
        )

        print('defining model group configuration')
        deployment_config = oci.data_science.models.ModelGroupDeploymentConfigurationDetails(
            deployment_type="MODEL_GROUP",
            model_group_configuration_details=model_group_config_details,
            infrastructure_configuration_details=infrastructure_config_details,
            environment_configuration_details=environment_config_details,
        )

        print('setting logging')
        category_log_details = oci.data_science.models.CategoryLogDetails(
            access=oci.data_science.models.LogDetails(log_group_id=LOG_GROUP_OCID, log_id=LOG_OCID),
            predict=oci.data_science.models.LogDetails(log_group_id=LOG_GROUP_OCID, log_id=LOG_OCID),
        )

        print('creating deploymnet...')
        response = client.create_model_deployment(
            oci.data_science.models.CreateModelDeploymentDetails(
                display_name=display_name,
                description=description,
                compartment_id=COMPARTMENT_OCID,
                project_id=PROJECT_OCID,
                model_deployment_configuration_details=deployment_config,
                category_log_details=category_log_details,
            )
        )
        deployment_id = response.data.id
        return wait_for_state(
            label=f"model deployment {display_name}",
            getter=lambda: client.get_model_deployment(deployment_id).data,
            success_states={"ACTIVE"},
            failure_states={"FAILED", "DELETED"},
            timeout_seconds=60 * 90,
        )
    except Exception as e:
        print(e)

In [10]:

def live_update_model_deployment(
    client: oci.data_science.DataScienceClient,
    model_deployment_id: str,
    display_name: str,
    description: str,
    new_model_group_id: str,
) -> oci.data_science.models.ModelDeployment:
    update_model_group_configuration_details = oci.data_science.models.UpdateModelGroupConfigurationDetails(
        model_group_id=new_model_group_id,
    )
    model_deployment_configuration_details = oci.data_science.models.UpdateModelGroupDeploymentConfigurationDetails(
        deployment_type="MODEL_GROUP",
        update_type="LIVE",
        model_group_configuration_details=update_model_group_configuration_details,
    )
    client.update_model_deployment(
        model_deployment_id=model_deployment_id,
        update_model_deployment_details=oci.data_science.models.UpdateModelDeploymentDetails(
            display_name=display_name,
            description=description,
            model_deployment_configuration_details=model_deployment_configuration_details,
        ),
    )
    return wait_for_state(
        label=f"model deployment update {model_deployment_id}",
        getter=lambda: client.get_model_deployment(model_deployment_id).data,
        success_states={"ACTIVE"},
        failure_states={"FAILED", "DELETED"},
        timeout_seconds=60 * 90,
    )


In [11]:
def _copy_file(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


def _copy_tree(src: Path, dst: Path) -> None:
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

In [12]:
def register_model(artifact_dir: Path,compartment_id,project_id,display_name) -> GenericModel:
    try:
        generic_model = GenericModel(artifact_dir=str(artifact_dir))
        print('model initialised')
        generic_model.prepare(
            inference_conda_env=INFERENCE_CONDA_ENV,
            inference_python_version=INFERENCE_PYTHON_VERSION,
            score_py_uri=str(TEMPLATES_DIR + "/single_model_score.py"),
            force_overwrite=True,
        )
        print('model prepared')
        _copy_file(Path(TEMPLATES_DIR) / "runtime.yaml", Path(artifact_dir) / "runtime.yaml")
        generic_model.reload_runtime_info()
        print('runtime.yaml updated')

        generic_model.save(compartment_id=compartment_id,project_id=project_id,display_name=display_name)
        print('model saved to OCI!')
        return generic_model
    except Exception as e:
        print(e)

### Create Model Pickle Files

In [13]:
# pickle model 1
serialize_model_instance(artifact_dir=Path(PROJECT_ROOT_PATH + 'complete-workflow-example/model1_live'),
                         package_name='square_service_v1_bundle',
                         class_name='SquareModel')

model serialised successfully


In [14]:
# pickle model 2
serialize_model_instance(artifact_dir=Path(PROJECT_ROOT_PATH + 'complete-workflow-example/model2_live'),
                         package_name='sqrt_service_v1_bundle',
                         class_name='SqrtModel')

model serialised successfully


In [15]:
# pickle model 3
serialize_model_instance(artifact_dir=Path(PROJECT_ROOT_PATH + 'complete-workflow-example/model3_live'),
                         package_name='abs_sqrt_service_v2_bundle',
                         class_name='AbsSqrtModel')

model serialised successfully


### Create Model Catalog Entries

In [ ]:
# model 1 catalog save
model1 = register_model(artifact_dir=Path(PROJECT_ROOT_PATH + 'complete-workflow-example/model1_live'),compartment_id=COMPARTMENT_OCID,project_id=PROJECT_OCID,display_name='Model 1')

In [ ]:
# model 2 catalog save
model2 = register_model(artifact_dir=Path(PROJECT_ROOT_PATH + 'complete-workflow-example/model2_live'),compartment_id=COMPARTMENT_OCID,project_id=PROJECT_OCID,display_name='Model 2')

In [ ]:
# model 3 catalog save
model3 = register_model(artifact_dir=Path(PROJECT_ROOT_PATH + 'complete-workflow-example/model3_live'),compartment_id=COMPARTMENT_OCID,project_id=PROJECT_OCID,display_name='Model 3')

### Create OCI SDK DataScience Client

In [19]:
# create client
config = oci.config.from_file("~/.oci/config")
ds_client = oci.data_science.DataScienceClient(config)

### Create Model Group 1

In [20]:
# model 1 + 2

model_group1 = create_model_group(client=ds_client,display_name='Model Group 1',description = 'Model 1 + Model 2',
                                  members=[{'model_id':model1.model_id,'inference_key':'square'},{'model_id':model2.model_id,'inference_key':'square-root'}],
                                  COMPARTMENT_OCID=COMPARTMENT_OCID,PROJECT_OCID=PROJECT_OCID,)

INFO:deploy_to_oci:model group Model Group 1 lifecycle_state=CREATING
INFO:deploy_to_oci:model group Model Group 1 lifecycle_state=CREATING
INFO:deploy_to_oci:model group Model Group 1 lifecycle_state=ACTIVE


### Create Model Group 2

In [21]:
# model 1 + 3

model_group2 = create_model_group(client=ds_client,display_name='Model Group 2',description = 'Model 1 + Model 3',
                                  members=[{'model_id':model1.model_id,'inference_key':'square'},{'model_id':model3.model_id,'inference_key':'square-root'}],
                                  COMPARTMENT_OCID=COMPARTMENT_OCID,PROJECT_OCID=PROJECT_OCID,)

INFO:deploy_to_oci:model group Model Group 2 lifecycle_state=CREATING
INFO:deploy_to_oci:model group Model Group 2 lifecycle_state=ACTIVE


### Add Model Group Artefact to Model Groups

NOTE: We use the *same* artefact for each model group

In [ ]:
upload_model_group_artifact(client=ds_client,model_group_id=model_group1,zip_path=Path('<full-path>/complete-workflow-example/model_group_live.zip'))

In [ ]:
upload_model_group_artifact(client=ds_client,model_group_id=model_group2,zip_path=Path('<full-path>/complete-workflow-example/model_group_live.zip'))

### Create Model Group Deployment (using Model Group 1)

In [24]:
# deploy model group 1
model_deployment = create_model_deployment(client=ds_client,
                                           display_name='Business Model Deployment',
                                           description='Model Deployment with custom logic',
                                           model_group_id=model_group1,
                                           LOG_GROUP_OCID=LOG_GROUP_OCID,
                                           LOG_OCID=LOG_OCID,
                                           COMPARTMENT_OCID=COMPARTMENT_OCID,
                                           PROJECT_OCID=PROJECT_OCID)

setting config
setting compute
setting scaling policy
setting concurrency
defining model group configuration
setting logging
creating deploymnet...
INFO:deploy_to_oci:model deployment Business Model Deployment lifecycle_state=CREATING
INFO:deploy_to_oci:model deployment Business Model Deployment lifecycle_state=CREATING
INFO:deploy_to_oci:model deployment Business Model Deployment lifecycle_state=CREATING
INFO:deploy_to_oci:model deployment Business Model Deployment lifecycle_state=CREATING
INFO:deploy_to_oci:model deployment Business Model Deployment lifecycle_state=CREATING
INFO:deploy_to_oci:model deployment Business Model Deployment lifecycle_state=CREATING
INFO:deploy_to_oci:model deployment Business Model Deployment lifecycle_state=CREATING
INFO:deploy_to_oci:model deployment Business Model Deployment lifecycle_state=CREATING
INFO:deploy_to_oci:model deployment Business Model Deployment lifecycle_state=CREATING
INFO:deploy_to_oci:model deployment Business Model Deployment lifecyc

### Score Model Group 1

In [25]:
import requests
import oci
from oci.signer import Signer

config = oci.config.from_file("~/.oci/config") # replace with the location of your oci config file
auth = Signer(
  tenancy=config['tenancy'],
  user=config['user'],
  fingerprint=config['fingerprint'],
  private_key_file_location=config['key_file'],
  pass_phrase=config['pass_phrase'])

endpoint = model_deployment.model_deployment_url + '/predict'
body = {'number': -9}
headers={'Content-Type':'application/json','model-key':'square-root'}
print(requests.post(endpoint, json=body, auth=auth, headers=headers).json())


{"code": "InternalServerError", "message": "math domain error", "status": "math domain error"}


### Live Update (using Model Group 2)

In [ ]:
updated_deployment = live_update_model_deployment(client=ds_client,model_deployment_id=model_deployment.id,display_name='Business Model Deployment',description='Updated deployment',new_model_group_id=model_group2)

### Score Model Group 2

In [29]:
endpoint = model_deployment.model_deployment_url + '/predict'
body = {'number': -9}
headers={'Content-Type':'application/json','model-key':'square-root'}
print(requests.post(endpoint, json=body, auth=auth, headers=headers).json())


{'prediction': 3.0}


The model deployment has been updated live and the function now returns the expected result.